In [1]:
from pathlib import Path
import polars as pl
import re

In [2]:
raw_dir = Path('../data/raw')
bronze_dir = Path('../data/bronze/buildings')

### **Check file count & file name**

In [3]:
raw_files = sorted(raw_dir.glob('*.csv.gz'))
bronze_files = sorted(bronze_dir.glob('*.parquet'))

raw_expected_bronze_names = {raw_file.name.replace('.csv.gz', '.parquet') for raw_file in raw_files}
actual_bronze_names = {bronze_file.name for bronze_file in bronze_files}

missing_bronze = sorted(raw_expected_bronze_names - actual_bronze_names)
unexpected_bronze = sorted(actual_bronze_names - raw_expected_bronze_names)

counts_match = 'Yes' if len(raw_files) == len(bronze_files) else 'No'
mapping_match = 'Yes' if not missing_bronze and not unexpected_bronze else 'No'

print(f'Raw file count: {len(raw_files)}')
print(f'Bronze file count: {len(bronze_files)}')
print(f'Counts match: {counts_match}')
print(f'One-to-one filename mapping: {mapping_match}')

Raw file count: 601
Bronze file count: 601
Counts match: Yes
One-to-one filename mapping: Yes


### Metadata value check

In [10]:
filename_pattern = re.compile(
      r'(?P<country>.+)_(?P<quadkey>\d+)_(?P<upload_date>\d{4}-\d{2}-\d{2})\.parquet$'
)

all_ok = True

for file in bronze_files:
    match = filename_pattern.fullmatch(file.name)
    if match is None:
        print(f'Bad bronze filename: {file.name}')
        all_ok = False
        continue

    expected = match.groupdict()
    expected['source_file'] = file.name.replace('.parquet', '.csv.gz')

    result = (
        pl.scan_parquet(file)
        .select(
            pl.len().alias('rows'),
            pl.col('country').n_unique().alias('country_n_unique'),
            pl.col('quadkey').n_unique().alias('quadkey_n_unique'),
            pl.col('upload_date').n_unique().alias('upload_date_n_unique'),
            pl.col('source_file').n_unique().alias('source_file_n_unique'),
            pl.col('country').drop_nulls().first().alias('country'),
            pl.col('quadkey').drop_nulls().first().alias('quadkey'),
            pl.col('upload_date').drop_nulls().first().alias('upload_date'),
            pl.col('source_file').drop_nulls().first().alias('source_file'),
        )
        .collect()
        .row(0, named=True)
    )

    if result['rows'] == 0:
        print(f'Empty bronze file: {file.name}')
        all_ok = False
        continue

    for col in ['country', 'quadkey', 'upload_date', 'source_file']:
        if result[f'{col}_n_unique'] != 1:
            print(f'{file.name}: {col} has {result[f'{col}_n_unique']} distinct values')
            all_ok = False
            continue

        if result[col] != expected[col]:
            print(f'{file.name}: {col} mismatch, expected {expected[col]!r}, got {result[col]!r}')
            all_ok = False

if all_ok:
    print('OK - all files passed')

OK - all files passed


### Geometry non-null and type check

In [11]:
valid_geometry_types = ['Polygon', 'MultiPolygon']

all_ok = True

for file in bronze_files:
    geometry_type = pl.col('geometry').struct.field('type')

    result = (
        pl.scan_parquet(file)
        .select(
            pl.col('geometry').null_count().alias('geometry_nulls'),
            geometry_type.null_count().alias('geometry_type_nulls'),
            geometry_type.filter(~geometry_type.is_in(valid_geometry_types)).len().alias('invalid_geometry_type_rows'),
            geometry_type.drop_nulls().unique().sort().alias('geometry_types'),
        )
        .collect()
        .row(0, named=True)
    )

    if result['geometry_nulls'] > 0:
        print(f'{file.name}: geometry has {result['geometry_nulls']} null rows')
        all_ok = False

    if result['geometry_type_nulls'] > 0:
        print(f'{file.name}: geometry.type has {result['geometry_type_nulls']} null rows')
        all_ok = False

    if result['invalid_geometry_type_rows'] > 0:
        print(
            f'{file.name}: found {result['invalid_geometry_type_rows']} rows with unexpected geometry types '
            f'{result['geometry_types']}'
        )
        all_ok = False

if all_ok:
    print('OK - all files passed')

OK - all files passed
